# Comparison of satellite images and learned embeddings for land cover mapping in the Brazilian Amazon 

## Models
1) Deep-learning approach: resnet50 trained on Sentinel-2 satellite images
2) Deep-learning approach: fully connected convnet trained on AE embeddings
3) Machine-learning approach: Random Forest classifier trained on a subset of pixels from AE embeddings

## Inference 
This notebook allows to run inference
1) Loads trained parameters from our models
2) Loads data from the test set
3) Applies the models on 5 images from the test set
4) Displays them

# Load Models and parameters

In [1]:
from utils import load_model_sentinel
from utils import load_model_AE
from utils import load_model_AE_RF

# Load models
model_s2=load_model_sentinel("models/s2/best_epoch4_fold4.pth")
model_AE=load_model_AE("models/AE/best.pth")
mdoel_AE_RF, scaler=load_model_AE_RF()

# Load 8 selected samples from the Test Set

In [14]:
import numpy as np
import rasterio

s2_path = "Test_for_inference/S2/"
ae_path = "Test_for_inference/AE/"
gt_path = "Test_for_inference/groundtruth/"

fnames = ["test_010040.tif", "test_010074.tif", "test_010087.tif", "test_010116.tif", "test_010143.tif", "test_010219.tif", "test_010351.tif", "test_010489.tif"]

LABEL_REMAP = {3:1, 4:2, 6:3, 9:4, 11:5, 12:6, 15:7, 18:8, 24:9, 25:10, 30:11, 33:12}
data = []
for fname in fnames:
    s2_img_path = s2_path + fname
    ae_img_path = ae_path + fname
    lbl_path = gt_path + fname
    # Read multispectral S2 image
    with rasterio.open(s2_img_path) as src:
        s2_img = src.read().astype(np.float32)  # (C, H, W)
    # Read AE embeddings
    with rasterio.open(ae_img_path) as src:
        ae_img = src.read().astype(np.float32)  # (C, H, W)
    # Read label mask (single channel)
    with rasterio.open(lbl_path) as src:
        label_raw = src.read(1).astype(np.int32)

    # Remap labels to contiguous indices
    label = np.zeros_like(label_raw, dtype=np.int32)
    for old_id, new_id in LABEL_REMAP.items():
        label[label_raw == old_id] = new_id
    
    data.append([s2_img, ae_img, label])

# Inference: run forward pass through each model

In [24]:
#from utils import inference
import numpy as np
import torch
@torch.no_grad() #without gradients
def inference(data, model, modality="s2", scaler=None):
    import numpy as np
    import torchvision.transforms as T
    # Only runs on CPU because of the limited amount of images to predict
    
    #normalization
    s2=np.load("mean_std/s2.npy") # Mean and std of Sentinel-2
    s2_mean, s2_std = s2[0], s2[1]
    s2_mean, s2_std = torch.tensor(s2_mean), torch.tensor(s2_std)
    s2_normalize = T.Normalize(s2_mean, s2_std)
    AE=np.load("mean_std/AE.npy") # Mean and std of AE-Embeddings
    AE_mean, AE_std = AE[0], AE[1]
    AE_mean, AE_std = torch.tensor(AE_mean), torch.tensor(AE_std)
    AE_normalize = T.Normalize(AE_mean, AE_std) 
    model.eval()
    predictions = []
    for id in range(8):
        #load image and ground truth
        s2_img, ae_img, label = data[id]
        #inference
        if modality == "s2":
            x = torch.from_numpy(s2_img).unsqueeze(0)
            x = s2_normalize(x)
            #forward pass
            outputs = model(x)
            pred = outputs["out"].argmax(1).squeeze(0).numpy().astype(np.uint8)
        elif modality == "AE":
            x = torch.from_numpy(ae_img).unsqueeze(0)
            x = AE_normalize(x)
            #forward pass
            pred = model(x)
            pred = pred.argmax(1).squeeze(0).numpy().astype(np.uint8)
        elif modality == "AE_RF":
            x = ae_img.astype(np.float32)
            x = scaler.transform(x)
            #prediction
            pred = model.predict(x)
        else:
            print("Modality must be 'AE', 'AE_RF' or 's2'")
        predictions.append(pred)
    return predictions

prediction_s2 = inference(data, model_s2, modality="s2")

KeyboardInterrupt: 